In [1]:
from transformers import AutoTokenizer, BertForSequenceClassification

# bert-base-uncased
# prajjwal1/bert-tiny

model_name = "prajjwal1/bert-tiny"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = BertForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2, 
    id2label={
        0: "Negative",
        1: "Positive"
    }
)

base_model.bert

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 128, padding_idx=0)
    (position_embeddings): Embedding(512, 128)
    (token_type_embeddings): Embedding(2, 128)
    (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-1): 2 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=128, out_features=128, bias=True)
            (key): Linear(in_features=128, out_features=128, bias=True)
            (value): Linear(in_features=128, out_features=128, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=128, out_features=128, bias=True)
            (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)


In [2]:
from datasets import load_dataset

def tokenize_function(examples):
    return tokenizer(examples["text"], max_length=128, padding="max_length", truncation=True)

dataset = load_dataset("fancyzhx/yelp_polarity")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [3]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(5000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

In [4]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [5]:
import torch
import torch.nn as nn
import copy

class ReSpropLinearFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, weight, bias, prev_grad_output, reuse_percentage):
        if prev_grad_output is not None and \
          len(input.shape) == 3 and prev_grad_output.size(0) == input.size(1):
            prev_grad_input  = torch.mm(prev_grad_output, weight)                       # pre∇w_l
            prev_grad_weight = torch.mm(prev_grad_output.t(), torch.sum(input, dim=0))  # pre∇a_l
        else:
            if prev_grad_output is not None:
                print("Warning: Couldn't reuse gradient due to shape mis-match.")
            
            prev_grad_output = None
            prev_grad_input  = None
            prev_grad_weight = None

        ctx.reuse_percentage = reuse_percentage
        ctx.save_for_backward(
            input, weight, bias, 
            prev_grad_output, prev_grad_input, prev_grad_weight
        )
        
        output = torch.bmm(input, weight.t().expand(input.size(0), -1, -1))
        if bias is not None:
            output += bias
        return output
    
    @staticmethod
    def backward(ctx, grad_output):
        input, weight, bias, prev_grad_output, prev_grad_input, prev_grad_weight = ctx.saved_tensors
        
        grad_input = None
        grad_weight = None
        grad_bias = None
        
        # Compute reuse mask
        if prev_grad_output is not None:
            grad_diff = grad_output - prev_grad_output

            if ctx.reuse_percentage > 0: 
                # Find threshold
                if grad_diff.device.type == "mps":
                    sorted_diffs = torch.sort(torch.abs(grad_diff).flatten())[0]
                    threshold_idx = int(len(sorted_diffs) * ctx.reuse_percentage)
                    threshold = sorted_diffs[threshold_idx]
                else:
                    threshold_idx = int(len(grad_diff.flatten()) * ctx.reuse_percentage)
                    threshold = torch.kthvalue(torch.abs(grad_diff).flatten(), threshold_idx)[0]

                # Sparsify grad_output
                reuse_mask = torch.abs(grad_diff) <= threshold
                grad_diff = torch.where(reuse_mask, torch.tensor(0), grad_diff)
                
            grad_output = grad_diff
        
        # Compute gradients
        if ctx.needs_input_grad[0]:
           grad_input = torch.bmm(grad_output, weight.expand(grad_output.size(0), -1, -1))
           if prev_grad_output is not None:
               grad_input += prev_grad_input

        if ctx.needs_input_grad[1]:
            grad_weight = torch.sum(torch.bmm(grad_output.transpose(1, 2), input), dim=0)
            if prev_grad_output is not None:
                grad_weight += prev_grad_weight

        if bias is not None and ctx.needs_input_grad[2]:
            grad_bias = grad_output.sum((0, 1))
        
        return grad_input, grad_weight, grad_bias, None, None


class ReSpropLinear(nn.Linear):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        bias: bool = True,
        device=None,
        dtype=None,
        reuse_percentage: float = 0.9
    ):
        super().__init__(in_features, out_features, bias, device, dtype)
        self.prev_gradients = None
        self.reuse_percentage = reuse_percentage
        
    def forward(self, input):
        output = ReSpropLinearFunction.apply(input, self.weight, self.bias, self.prev_gradients, self.reuse_percentage)
        
        if output.requires_grad:
            def hook(grad_output):
                # Store gradients for next iteration
                self.prev_gradients = grad_output[torch.randint(0, grad_output.size(0), (1,))][0].clone().detach()
                return None
            output.register_hook(hook)
        else:
            self.prev_gradients = None
            
        return output
    
    def extra_repr(self):
        return super().extra_repr() + f", reuse_percentage={self.reuse_percentage}"
    
def resprofify_bert(base_model, reuse_percentage=0.9):
    def resprop_linear(layer: nn.Linear):
        return ReSpropLinear(
            layer.in_features, 
            layer.out_features, 
            layer.bias is not None,
            reuse_percentage=reuse_percentage
        )
        
    model = copy.deepcopy(base_model)
    for layer in model.bert.encoder.layer:
        # Self Attention
        att = layer.attention
        att.self.query   = resprop_linear(att.self.query)
        att.self.key     = resprop_linear(att.self.key)
        att.self.value   = resprop_linear(att.self.value)
        att.output.dense = resprop_linear(att.output.dense)
        
        # Feed Forward Block
        layer.intermediate.dense = resprop_linear(layer.intermediate.dense)
        layer.output.dense       = resprop_linear(layer.output.dense)
        
    return model

In [6]:
model = resprofify_bert(base_model, reuse_percentage=0.9)

# Freeze Bert
# model.bert.requires_grad_(False)

In [7]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(output_dir="trainer_out", eval_strategy="epoch")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)


In [8]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.675700,0.559606,0.749000
2,0.534700,0.390114,0.838000
3,0.432200,0.379651,0.843000


TrainOutput(global_step=1875, training_loss=0.5152171468098958, metrics={'train_runtime': 152.9862, 'train_samples_per_second': 98.048, 'train_steps_per_second': 12.256, 'total_flos': 4764326400000.0, 'train_loss': 0.5152171468098958, 'epoch': 3.0})

In [9]:
inputs = tokenizer("love this place!", return_tensors="pt")

with torch.no_grad():
    logits = model(
        **{k : v.to(model.device) for k, v in inputs.items() }
    ).logits

predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]

'Positive'

In [10]:
trainer.evaluate()

{'eval_loss': 0.37965089082717896,
 'eval_accuracy': 0.843,
 'eval_runtime': 1.6215,
 'eval_samples_per_second': 616.694,
 'eval_steps_per_second': 77.087,
 'epoch': 3.0}

In [12]:
model = copy.deepcopy(model)
model.to(torch.device("mps"))

loss_fct = nn.CrossEntropyLoss()
dataloader = torch.utils.data.DataLoader(small_train_dataset, batch_size=16, shuffle=False)
data = next(iter(dataloader))
inputs = dict(
    input_ids = torch.stack(data['input_ids'], dim=1).to(model.device),
    token_type_ids = torch.stack(data['token_type_ids'], dim=1).to(model.device),
    attention_mask = torch.stack(data['attention_mask'], dim=1).to(model.device),
)
labels = torch.tensor([1] * dataloader.batch_size).to(model.device)

# Warm Up
logits = model(**inputs).logits
loss = loss_fct(logits.view(-1, 2), labels.view(-1))
loss.backward()

with torch.profiler.profile() as prof:
    for _ in range(5):
        logits = model(**inputs).logits
        loss = loss_fct(logits.view(-1, 2), labels.view(-1))
        loss.backward()

In [13]:
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=20, max_name_column_width=100))
print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=20, max_name_column_width=100))

------------------------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                                    Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
------------------------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
      autograd::engine::evaluate_function: ReSpropLinearFunctionBackward         0.55%       2.069ms        67.41%     251.562ms       4.193ms            60  
                                           ReSpropLinearFunctionBackward         1.08%       4.012ms        58.90%     219.793ms       3.663ms            60  
                                                             aten::copy_        46.23%     172.543ms        46.28%     172.721ms     431.802us           400  
                                              

In [14]:
# Inspect trace using magic-trace.org
prof.export_chrome_trace("./out.trace")